# Base e recuperação do Reencontro

Este notebook executa a conferência da base e repete a busca das perguntas congeladas. Não executa a LLM. A base contém sete documentos textuais, separados dos 100 casos fictícios do portal. As saídas foram calculadas a partir dos arquivos recebidos, sem criação de novos resultados de treinamento.


In [1]:
from pathlib import Path
import json
import sys

RAIZ = Path.cwd()
if not (RAIZ / "base" / "conhecimento.json").exists():
    RAIZ = RAIZ.parent
assert (RAIZ / "base" / "conhecimento.json").exists(), "Abra a partir da raiz do projeto ou da pasta notebooks."
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

def ler_json(caminho):
    return json.loads((RAIZ / caminho).read_text(encoding="utf-8-sig"))


In [2]:
base = ler_json("base/conhecimento.json")
assert len({item["id"] for item in base}) == len(base)
print("Trechos consultáveis:", len(base))
for item in base:
    print(item["id"], "|", item["titulo"], "|", item["fonte"])
print("\nFontes e condições de reutilização")
for item in ler_json("base/fontes.json"):
    print(item["fonte"], "|", item["numero_trechos"], "trechos |", item["licenca"])


Trechos consultáveis: 7
oficial_001 | Registro do desaparecimento | Senado Federal — Pessoas Desaparecidas
oficial_002 | Comunicação pelo SINAL DESAPARECIDOS | Portal Gov.br — SINAL DESAPARECIDOS
produto_001 | Finalidade do Reencontro | Documentação do projeto Reencontro
produto_002 | Consulta por informações | Documentação do projeto Reencontro
produto_003 | Busca por foto | Documentação do projeto Reencontro
produto_004 | Envio de informações adicionais | Documentação do projeto Reencontro
produto_005 | Limites das informações disponíveis | Documentação do projeto Reencontro

Fontes e condições de reutilização
Senado Federal | 1 trechos | Não identificada na página consultada. Confirmar condições de reutilização.
Portal Gov.br e PRF | 1 trechos | Rodapé informa CC BY-ND 3.0 Não Adaptada.
Documentação autoral Reencontro | 5 trechos | Licença de redistribuição a definir pela autora.


## Calibração registrada

O limiar foi escolhido em 14 perguntas de desenvolvimento. A saída a seguir é lida do arquivo de calibração original. Não é um novo ajuste de limiar.


In [3]:
calibracao = ler_json("avaliacao/calibracao_busca.json")
print(json.dumps(calibracao["melhor_configuracao"], ensure_ascii=False, indent=2))
print(json.dumps(calibracao["metricas_recuperacao"], ensure_ascii=False, indent=2))


{
  "limiar": 0.2,
  "sensibilidade": 1.0,
  "especificidade": 1.0,
  "acuracia_balanceada": 1.0,
  "respondiveis_aceitas": 7,
  "respondiveis_rejeitadas": 0,
  "sem_resposta_aceitas": 0,
  "sem_resposta_rejeitadas": 7
}
{
  "recall_1_antes": 1.0,
  "recall_3_antes": 1.0,
  "recall_1_depois": 1.0,
  "recall_3_depois": 1.0
}


## Reexecução da recuperação

A célula executa TF-IDF com limiar 0,20 e compara o ranking aos IDs gravados na avaliação 03. A biblioteca instalada nesta análise pode diferir da versão original, por isso os IDs são conferidos explicitamente. Não altera os arquivos congelados.


In [4]:
from busca import BuscaConhecimento
import sklearn

print("Scikit-learn nesta execução:", sklearn.__version__)
busca = BuscaConhecimento(limiar=0.20)
avaliacao = ler_json("avaliacao/avaliacao_congelada_20261114_160302_507373.json")
recalls = []
for caso in avaliacao["resultados"]:
    recuperados = busca.buscar(caso["pergunta"], quantidade=3)
    ids = [item["id"] for item in recuperados]
    ids_originais = [item["id"] for item in caso["documentos_recuperados"]]
    print(caso["id"], ids, "| ranking igual ao registrado:", ids == ids_originais)
    if caso["fontes"]:
        esperadas = set(caso["fontes"])
        recalls.append((len(esperadas.intersection(ids[:1])) / len(esperadas), len(esperadas.intersection(ids[:3])) / len(esperadas)))
print("\nPerguntas respondíveis:", len(recalls))
print("Recall@1:", round(sum(x[0] for x in recalls) / len(recalls), 4))
print("Recall@3:", round(sum(x[1] for x in recalls) / len(recalls), 4))


Scikit-learn nesta execução: 1.8.0
j001 ['oficial_001'] | ranking igual ao registrado: True
j002 ['oficial_002'] | ranking igual ao registrado: True
j003 [] | ranking igual ao registrado: True
j004 ['produto_002', 'produto_003'] | ranking igual ao registrado: True
j005 ['produto_003'] | ranking igual ao registrado: True
j006 ['produto_004'] | ranking igual ao registrado: True
j007 ['oficial_001', 'produto_003'] | ranking igual ao registrado: True
j008 ['produto_001'] | ranking igual ao registrado: True
j009 [] | ranking igual ao registrado: True
j010 [] | ranking igual ao registrado: True
j011 [] | ranking igual ao registrado: True
j012 ['produto_001'] | ranking igual ao registrado: True

Perguntas respondíveis: 6
Recall@1: 0.8333
Recall@3: 0.8333


## Interpretação

A calibração perfeita não se manteve nas reformulações. O limiar não prova que um documento responde à pergunta. Uma busca vazia também não prova que a informação inexiste na base. A pergunta j003 demonstra uma falsa rejeição. A API encerra com recusa nesse caso, ao passo que o avaliador executou o modelo para registrar o comportamento.
